In [1]:
from pathlib import Path
import pandas as pd
import subprocess
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

PROJECT_ROOT = Path("/home/jovyan/MSC_PROJECT")
DATA_DIR = PROJECT_ROOT / "data" / "avdeepfake1mpp"
EXTRACTED_VAL = DATA_DIR / "extracted" / "val" / "val"
MANIFEST_DIR = DATA_DIR / "manifests"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

videos = sorted(EXTRACTED_VAL.rglob("*.mp4"))

print("Videos found:", len(videos))
print(videos[0])

Videos found: 77326
/home/jovyan/MSC_PROJECT/data/avdeepfake1mpp/extracted/val/val/lrs3/1Hok1iGFNSk/00004/fake_video_fake_audio.mp4


In [2]:
def probe_audio(video_path):
    video_path = Path(video_path)

    cmd = [
        "ffprobe",
        "-v", "error",
        "-select_streams", "a:0",
        "-show_entries", "stream=codec_name,sample_rate,channels,duration",
        "-of", "json",
        str(video_path)
    ]

    process = subprocess.run(cmd, capture_output=True, text=True)

    row = {
        "path": str(video_path),
        "relative_path": str(video_path.relative_to(EXTRACTED_VAL)),
        "filename": video_path.name,
        "file_size_bytes": video_path.stat().st_size if video_path.exists() else None,
        "ffprobe_returncode": process.returncode,
        "has_audio_stream": False,
        "audio_codec": None,
        "sample_rate": None,
        "channels": None,
        "audio_duration": None,
        "ffprobe_error": process.stderr[-500:] if process.stderr else None,
    }

    if process.returncode != 0:
        return row

    try:
        data = json.loads(process.stdout)
        streams = data.get("streams", [])

        if len(streams) == 0:
            return row

        stream = streams[0]

        row["has_audio_stream"] = True
        row["audio_codec"] = stream.get("codec_name")
        row["sample_rate"] = stream.get("sample_rate")
        row["channels"] = stream.get("channels")
        row["audio_duration"] = stream.get("duration")

    except Exception as e:
        row["ffprobe_error"] = str(e)

    return row

In [3]:
def test_audio_decode(video_path, seconds=1):
    video_path = Path(video_path)

    cmd = [
        "ffmpeg",
        "-v", "error",
        "-nostdin",
        "-i", str(video_path),
        "-map", "0:a:0",
        "-t", str(seconds),
        "-f", "null",
        "-"
    ]

    process = subprocess.run(cmd, capture_output=True, text=True)

    return {
        "audio_decode_returncode": process.returncode,
        "audio_decodable": process.returncode == 0,
        "audio_decode_error": process.stderr[-500:] if process.stderr else None,
    }

In [4]:
def audit_one_video(video_path, decode_seconds=1):
    row = probe_audio(video_path)

    if row["has_audio_stream"]:
        decode_result = test_audio_decode(video_path, seconds=decode_seconds)
        row.update(decode_result)
    else:
        row.update({
            "audio_decode_returncode": None,
            "audio_decodable": False,
            "audio_decode_error": "No audio stream found"
        })

    return row

In [ ]:
AUDIO_AUDIT_PATH = MANIFEST_DIR / "audio_audit_all_videos.csv"

rows = []

max_workers = 8
decode_seconds = 1

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {
        executor.submit(audit_one_video, video_path, decode_seconds): video_path
        for video_path in videos
    }

    for future in tqdm(as_completed(futures), total=len(futures)):
        try:
            rows.append(future.result())
        except Exception as e:
            video_path = futures[future]
            rows.append({
                "path": str(video_path),
                "relative_path": str(video_path.relative_to(EXTRACTED_VAL)),
                "filename": video_path.name,
                "has_audio_stream": False,
                "audio_decodable": False,
                "fatal_error": str(e),
            })

audio_audit_df = pd.DataFrame(rows)
audio_audit_df.to_csv(AUDIO_AUDIT_PATH, index=False)

print("Saved:", AUDIO_AUDIT_PATH)
audio_audit_df.head()

 33%|███▎      | 25373/77326 [12:03<26:37, 32.52it/s]

In [ ]:
audio_audit_df = pd.read_csv(AUDIO_AUDIT_PATH)

print("Total videos:", len(audio_audit_df))

print("\nHas audio stream:")
print(audio_audit_df["has_audio_stream"].value_counts(dropna=False))

print("\nAudio decodable:")
print(audio_audit_df["audio_decodable"].value_counts(dropna=False))

print("\nSample rates:")
print(audio_audit_df["sample_rate"].value_counts(dropna=False).head(20))

print("\nChannels:")
print(audio_audit_df["channels"].value_counts(dropna=False).head(20))

print("\nCodecs:")
print(audio_audit_df["audio_codec"].value_counts(dropna=False).head(20))

In [ ]:
failed_audio_df = audio_audit_df[
    (audio_audit_df["has_audio_stream"] != True) |
    (audio_audit_df["audio_decodable"] != True)
].copy()

FAILED_AUDIO_PATH = MANIFEST_DIR / "audio_audit_failed_videos.csv"
failed_audio_df.to_csv(FAILED_AUDIO_PATH, index=False)

print("Failed videos:", len(failed_audio_df))
print("Saved:", FAILED_AUDIO_PATH)

failed_audio_df.head(20)